# 10 — Model-response and compute analysis

This notebook implements `analysis/NOTEBOOK_10_ANALYSIS_SPECIFICATION.md`. It has three aims:

1. analyse model responses using the already-materialized Haiku judge fields;
2. report collected compute/API usage while separating inference from training, preparation, restoration and transfer; and
3. analyse compute relationships without treating backend-confounded comparisons as causal effects.

The notebook **does not reparse raw model or judge responses**. It reads canonical exports under `results/final/` and writes only to `analysis/tables/`, `analysis/figures/` and `analysis/analysis_manifest.json`.

Population policy is explicit: `evaluation_285` is the primary like-for-like population for treatment, representation, teacher, baseline and non-inferiority comparisons; `all_990` preserves the full untreated universe as a sensitivity analysis. Main comparison figures use `evaluation_285`, while `_990_` figures provide untreated-only full-universe views. MDE/non-inferiority and Qwen-family size tables contain the applicable 990 sensitivity rows. Non-inferiority is marked floor-limited when its BF16 reference is at or below the fixed modal/null trivial-baseline anchor or at or below the fixed 0.3 margin; at least 15 paired scored rows are required for an interpretable verdict. Near-perfect low-score overlap is also not estimable when both models score at least 95% of common rows at 0–1. Compute session totals remain full-stream accounting; matched evaluation-pool rates are derived separately from completed row-performance records.

Visual conventions are fixed across the notebook: model colours identify model families; adapter colours/markers identify conditions; BF16 HF, TQ1_0 and native I2_S remain explicitly labelled. Separate BF16 and one-bit-frontier figures use identical axes where direct comparison is intended. **Every figure includes an explicit key**, and every saved figure is registered in `analysis/tables/figure_manifest.csv` with a companion plotted-data CSV under `analysis/tables/figure_data/`.

### Restart guidance

Each numbered execution section is idempotent: it replaces only its own derived tables/figures. After an interruption, rerun Section 1, then the interrupted section and all later sections. Source result CSVs are never modified.


## Section 1 — Local setup and frozen inputs

Use the project `.venv-local` kernel. Required analysis packages are listed in `requirements-analysis.txt`.

**If interrupted:** rerun this section before any later section. It only imports code, locates the repository and loads canonical CSVs.


In [11]:
from pathlib import Path
import os
import sys

STARTING_DIRECTORY = Path.cwd().resolve()
KNOWN_LOCAL_ROOT = Path(r'D:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment')
candidates = [STARTING_DIRECTORY, *STARTING_DIRECTORY.parents, KNOWN_LOCAL_ROOT]
ROOT = next((p.resolve() for p in candidates if (p / 'results/final').is_dir() and (p / 'analysis/NOTEBOOK_10_ANALYSIS_SPECIFICATION.md').is_file()), None)
if ROOT is None:
    checked = '\n'.join(f'  - {p}' for p in candidates)
    raise RuntimeError('Could not locate streamlined_experiment. Checked:\n' + checked)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.notebook10_analysis import Notebook10Analysis, BOOTSTRAP_REPLICATES

bootstrap_replicates = int(os.environ.get('CAM_LDS_ANALYSIS_BOOTSTRAPS', BOOTSTRAP_REPLICATES))
analysis = Notebook10Analysis(ROOT, bootstrap_replicates=bootstrap_replicates)
print('Experiment:', ROOT)
print('Judgment rows:', len(analysis.judgments))
print('Response rows:', len(analysis.responses))
print('Bootstrap replicates:', analysis.bootstrap_replicates)


Experiment: D:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment
Judgment rows: 25516
Response rows: 25425
Bootstrap replicates: 10000


## Section 2 — Reproducibility manifest and score eligibility

Creates the initial input-hash manifest, distinguishes absent responses from unavailable scores, and reports canonical scalar-score coverage. Score objects, malformed strings and nulls remain unavailable; they are never converted to score zero.

**If interrupted:** rerun Sections 1–2. This section is read-only with respect to `results/final/`.


In [3]:
analysis.build_manifest(final=False)
analysis.availability()
print('Wrote score-availability tables, identification bounds and Figure 00.')


Wrote score-availability tables, identification bounds and Figure 00.


## Section 3 — Multidimensional score summaries

Calculates independent summaries for every rubric dimension, the full ordinal distributions and scenario-step cluster-bootstrap intervals. It generates separate BF16 and one-bit figures on common scales. Untreated/full figures are the main comparison; conditional adapters are supplementary.

**If interrupted:** rerun Section 1 and this section. Rerunning Section 2 is recommended if canonical inputs changed.


In [4]:
analysis.score_summaries()
print('Wrote score summaries, ordinal distributions, scale-use diagnostics and Figures 01–02/S02.')


Wrote score summaries, ordinal distributions, scale-use diagnostics and Figures 01–02/S02.


## Section 4 — Exact common-support treatment contrasts

Matches identical prompt IDs before calculating SFT, deployment-stack and difference-family effects. It reports paired differences, transitions, probability of superiority, clustered intervals, sign-flip tests and BH-adjusted p-values. Deployment-stack results are explicitly not interpreted as pure quantization effects.

**If interrupted:** rerun Section 1 and Section 4. Section 3 is not computationally required, but should normally precede it for interpretation.


In [5]:
analysis.contrasts()
print('Wrote common-support diagnostics, contrast tables and Figures 03–04.')


Wrote common-support diagnostics, contrast tables and Figures 03–04.


## Section 5 — ATT&CK classes, related dimensions and failure analysis

Describes technique/parent-technique distributions and class-associated judge scores without parsing candidate text. Tactic output is marked unavailable unless a pinned ATT&CK STIX mapping exists in the frozen experiment. Failure rates use normalized denominators; raw counts remain in companion tables.

**If interrupted:** rerun Section 1 and this section.


In [6]:
analysis.attack_and_failures()
analysis.related_dimensions()
print('Wrote ATT&CK, multiplicity, related-dimension and failure outputs (Figures 05–10/S06–S07).')


Wrote ATT&CK, multiplicity, related-dimension and failure outputs (Figures 05–10/S06–S07).


## Section 6 — Repeated prompts, judge diagnostics, long tails and model similarity

Runs the predeclared internal-consistency and bias diagnostics, TF-IDF reference-similarity appendix, prompt-length analysis and matched item-difficulty concordance. These outputs do not establish judge reliability or semantic correctness.

**If interrupted:** rerun Section 1 and this section. This is the longest response-analysis section.


In [7]:
analysis.repeat_judge_longtail_size()
analysis.stopping_runtime()
print('Wrote repeated-prompt, judge-diagnostic, long-tail, stopping and concordance outputs (Figures 11–18).')


Wrote repeated-prompt, judge-diagnostic, long-tail, stopping and concordance outputs (Figures 11–18).


## Section 6A — Reference ladder, retention, size and distillation fidelity

Implements specification Sections 1.14–1.18 using the separately judged GPT-5.5 teacher and trivial reference streams from Notebook 7. It writes `reference_ladder.csv`, `retention_index.csv`, `size_contrasts.csv`, `mde_and_noninferiority.csv` and `distillation_fidelity.csv`, plus additive Figures 16C, 19A and 19B. Existing figures and numbering remain unchanged.

This section requires Notebook 7's teacher and baseline ledgers to be complete and Notebook 8 to have refreshed `results/final`. Reference rows use the same scalar-score eligibility rule as every model candidate.

**If interrupted:** rerun Section 1 and this section. It is read-only with respect to experimental source results.


In [13]:
import importlib
notebook10_module = importlib.import_module('analysis.notebook10_analysis')
notebook10_module = importlib.reload(notebook10_module)
analysis = notebook10_module.Notebook10Analysis(ROOT, bootstrap_replicates=bootstrap_replicates)
analysis.reference_extensions()
print('Wrote reference ladder, retention, Qwen-family size, MDE/non-inferiority and distillation-fidelity outputs (Figures 16C and 19A–19B).')

d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\.venv-local\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1593: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a,


Wrote reference ladder, retention, Qwen-family size, MDE/non-inferiority and distillation-fidelity outputs (Figures 16C and 19A–19B).


## Section 6B — Judge-mediated ranked-technique accuracy

Calculates exact-ID Hit@1 through Hit@10 and fixed-rank Precision@k, macro Recall@k and micro Recall@k at k = 1, 3, 5 and 10 from the canonical `judge_extracted_predictions` and `ground_truth` arrays. It also performs a separately labelled parent-normalized sensitivity analysis in which `Txxxx.xxx` is collapsed to `Txxxx`; exact-ID matching remains primary. It does not inspect or reparse raw candidate or judge prose. Explicit `[]` is a valid no-prediction result and scores zero; missing, null and malformed extraction fields remain unavailable in the primary analysis and are included only in the documented missing-as-zero sensitivity estimate.

Outputs include a row-level audit containing exact and parent-normalized values, extraction coverage, every model/adapter/representation stream, exact common-support contrasts, a core-stream Hit@k figure, supplementary all-adapter heatmaps and an exact-versus-parent sensitivity figure. The metrics are explicitly judge-mediated and do not replace `score_technique_identification`.

**If interrupted:** rerun Section 1 and this section. Section 6A is not a prerequisite once the canonical judgment CSVs are present. This section performs no API requests or model inference.


In [ ]:
import importlib
notebook10_module = importlib.import_module('analysis.notebook10_analysis')
notebook10_module = importlib.reload(notebook10_module)
analysis = notebook10_module.Notebook10Analysis(ROOT, bootstrap_replicates=bootstrap_replicates)
rank_outputs = analysis.judge_mediated_rank_metrics()
print({name: frame.shape for name, frame in rank_outputs.items()})
print('Wrote exact and parent-normalized judge-mediated Hit@1–10, Precision@k and Recall@k outputs plus Figures 19C and S19C–S19F.')

## Section 6C — Representativeness of the 604-row training subset

Audits the universal training exclusion caused by complete examples exceeding at least one model's training context. It compares the 604 retained and 101 excluded rows by raw prompt length, every model tokenizer, scenario and scenario-step group, and reports which model limits produced the exclusions. This is descriptive sensitivity evidence: it does not alter, reweight or retrain any model.

Outputs include a row audit, model-feasibility table, length-comparison table, scenario table, scenario-step coverage table and Figure 18F.

**If interrupted:** rerun Section 1 and this section. It requires only the frozen local `data/` files and performs no API requests or model inference.


In [ ]:
import importlib
notebook10_module = importlib.import_module('analysis.notebook10_analysis')
notebook10_module = importlib.reload(notebook10_module)
analysis = notebook10_module.Notebook10Analysis(ROOT, bootstrap_replicates=bootstrap_replicates)
training_subset_outputs = analysis.training_subset_representativeness()
print({name: frame.shape for name, frame in training_subset_outputs.items()})
print('Wrote the 604-versus-101 training-subset audit and Figure 18F.')

## Section 6D — Unseen techniques and split balance

Compares exact ground-truth technique coverage between the 604-row feasible training subset and the 285-row evaluation pool. It also reports distinct prompt counts and compares scenario composition, label multiplicity and prompt-length distributions across original train, feasible train, validation, test and the combined evaluation pool.

Outputs include population, scenario, multiplicity and prompt-length tables; exact technique-overlap and affected-row tables; and Figure 18G. This is a descriptive data audit and does not change any split or response.

**If interrupted:** rerun Section 1 and this section. It requires only frozen local `data/` files and performs no API requests or model inference.


In [12]:
import importlib
notebook10_module = importlib.import_module('analysis.notebook10_analysis')
notebook10_module = importlib.reload(notebook10_module)
analysis = notebook10_module.Notebook10Analysis(ROOT, bootstrap_replicates=bootstrap_replicates)
split_balance_outputs = analysis.split_label_balance()
print({name: frame.shape for name, frame in split_balance_outputs.items()})
print('Wrote unseen-technique and split-balance tables plus Figure 18G.')

{'population_summary': (5, 13), 'scenario_balance': (33, 4), 'label_multiplicity': (20, 4), 'prompt_length': (45, 9), 'technique_overlap': (93, 6), 'unseen_rows': (60, 10)}
Wrote unseen-technique and split-balance tables plus Figure 18G.


## Section 6E — Positive one-bit rows and training-label exposure

Creates a complete qualitative register of every one-bit-frontier judgment row with at least one score above zero. The register preserves exact stream and prompt identity, every canonical score, judge-derived labels, response hash, and a 600-character whitespace-normalized candidate-response excerpt. It does not select only extreme examples and therefore includes native-I2_S BitNet rows whenever they qualify.

It also joins the existing unseen-technique audit to all 285 evaluation prompts. Rows are separated into all labels seen in the 604-row training subset, mixed seen/unseen labels, or all labels unseen. Score summaries and paired SFT-minus-untreated effects are reported separately for each exposure stratum, model, representation, adapter and judge dimension. This describes exact-label exposure; it does not claim that a pretrained model had never encountered the technique.

**If interrupted:** rerun Section 1, Section 6D, and this section. No API calls or model inference are performed.


In [ ]:
import importlib
notebook10_module = importlib.import_module('analysis.notebook10_analysis')
notebook10_module = importlib.reload(notebook10_module)
analysis = notebook10_module.Notebook10Analysis(ROOT, bootstrap_replicates=bootstrap_replicates)
frontier_exposure_outputs = analysis.positive_frontier_and_training_exposure()
print({name: frame.shape for name, frame in frontier_exposure_outputs.items()})
print('Wrote the one-bit positive-score register and training-label exposure analyses.')


## Section 6F — Deterministic BitNet template example and prevalence

Selects P1 and P2 by the predeclared rule from canonical prompt, response and judgment CSVs only. It verifies prompt SHA-256 values, requires completed untreated BF16 and native-I2_S responses with judgments, prefers a natural BF16 ending, chooses the shortest eligible prompt, and requires P2 to come from another scenario. No response is regenerated or re-judged.

The final regex covers the two observed degenerate opening forms: confidence-list enumeration and direct echoing of the requested `Techniques` field. The output reports the actual prevalence; it does not force the result to support an ‘almost all’ claim.

**If interrupted:** rerun Section 1, Section 6E if the positive-score register is absent, then this cell.


In [ ]:
import importlib
notebook10_module = importlib.import_module('analysis.notebook10_analysis')
notebook10_module = importlib.reload(notebook10_module)
analysis = notebook10_module.Notebook10Analysis(ROOT, bootstrap_replicates=bootstrap_replicates)
bitnet_template_outputs = analysis.bitnet_template_example()
analysis.finalize()
print('P1:', bitnet_template_outputs['P1'], 'P2:', bitnet_template_outputs['P2'])
print('Natural-BF16 fallback used:', bitnet_template_outputs['natural_bf16_condition_dropped'])
print('Final regex:', bitnet_template_outputs['regex'])
display(bitnet_template_outputs['prevalence'])


## Section 7 — Compute-usage reporting

Reports every collected telemetry invocation and preserves retries/resumes as consumed resources. Training, preparation/upload, restoration/load, experimental inference, pilots and API use remain distinct. Invocation wall time is labelled GPU-runtime session time, not active GPU compute.

This section now reconstructs phase and exact-stream summaries from Notebook 9's **invocation inventory, row-performance and coverage source files**, rather than relying only on Notebook 9's flattened summaries. It also reads saved training logs for loss histories and quantized `artifact.json` manifests for artifact bytes, SHA-256 identities and preparation metadata. `compute_source_reconciliation.csv` records the comparison with Notebook 9 totals.

For training-loss histories, later no-op resume checks may overwrite a working log with only `adapter already complete`. The extractor therefore searches both live logs and `telemetry/logs/recovered/`, selects the source with the greatest number of loss observations for each model/adapter, and records `source_log` plus `source_provenance`. The four BitNet histories are recovered byte-for-byte from Git commit `8ec2bad28`; this restores 114 logged observations per adapter without estimating or rerunning training.

**If interrupted:** rerun Section 1 and this section.


In [8]:
analysis.compute_reporting()
print('Wrote compute audit/report tables and Figures 20–26.')


d:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\.venv-local\Lib\site-packages\matplotlib\transforms.py:2496: RuntimeWarning: invalid value encountered in dot
  return Affine2D(np.dot(self._b.get_affine().get_matrix(),


Wrote compute audit/report tables and Figures 20–26.


## Section 8 — Compute analysis

Creates descriptive size/efficiency, deployment-stack, cost/outcome, overhead and runtime-configuration/frontier analyses from the source-derived compute tables created in Section 7. It also reconciles observed invocation telemetry against slot-adjusted row timing, timestamp-union sensitivity and known timeout attempts for every exact stream. Observed telemetry is retained wherever complete; reconstruction fills inference gaps only and never invents loading, restoration, upload, idle, utilization, memory or energy values. Genuine inference-session time remains separate from training, artifact preparation/upload, model restoration/load, checkpointing and receiver overhead. Hardware/backend mismatches remain visible and prevent causal speed-up claims.

**If interrupted:** rerun Sections 1, 7 and 8.


In [9]:
analysis.compute_analysis()
print('Wrote compute-analysis tables and Figures 30–37, including runtime reconciliation.')


Wrote compute-analysis tables and Figures 30–37, including runtime reconciliation.


## Section 9 — Finalize and audit outputs

Hashes every generated artifact and rewrites the final analysis manifest. Run this after all desired sections complete.

**If interrupted:** rerun Sections 1 and 9. If any earlier section failed, complete it before finalizing.


In [14]:
manifest = analysis.finalize()
catalog = analysis.p.tables / 'analysis_output_catalog.csv'
print('Final manifest:', analysis.p.analysis / 'analysis_manifest.json')
print('Output catalog:', catalog)
print('Hashed outputs:', len(manifest['outputs']))


Final manifest: D:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\analysis\analysis_manifest.json
Output catalog: D:\reit4841_code\1-58-bit-llm-CAM-LDS\streamlined_experiment\analysis\tables\analysis_output_catalog.csv
Hashed outputs: 481


## Interpretation boundaries

- No judge dimensions are combined into a headline score.
- Validation and test provenance is retained, but the established 285-row evaluation pool is used for treatment comparisons.
- Partial streams use exact common support; available-case summaries are descriptive.
- Cluster-bootstrap intervals condition on the observed trained adapter, artifact, runtime history and single Haiku judgment pass.
- Haiku self-evaluation is a circular descriptive baseline, not judge validation.
- TQ1_0 and native I2_S belong to a broad one-bit-frontier overview but remain distinct representations.
- BF16-to-GGUF comparisons include inference-backend and scheduling differences.
- Deterministic format columns are secondary diagnostics and are not substituted for the Haiku judge.
